In [ ]:
import subprocess, sys
for pkg in ["pyyaml", "tqdm"]:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], capture_output=True)

In [ ]:
import os, sys, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
import yaml
from tqdm import tqdm

warnings.filterwarnings('ignore')

print(f"Python  : {sys.version.split()[0]}")
print(f"NumPy   : {np.__version__}")
print(f"Pandas  : {pd.__version__}")
print(f"OpenCV  : {cv2.__version__}")

In [ ]:
# =============================================================================
# ENVIRONMENT SWITCH
# Change to 'local' when running on your local machine.
# =============================================================================

ENVIRONMENT = "kaggle"  # <-- change to 'local' when running locally

if ENVIRONMENT == "kaggle":
    YOLO_DIR = Path("/kaggle/input/datasets/lokisilvres/dental-disease-panoramic-detection-dataset/YOLO/YOLO")

elif ENVIRONMENT == "local":
    import os as _os
    _base    = _os.path.join(_os.path.expanduser("~"), "Downloads", "Graduation", "Dental X-Ray dataset")
    YOLO_DIR = Path(_base) / "YOLO" / "YOLO"

else:
    raise ValueError(f"Unknown ENVIRONMENT: {ENVIRONMENT}")

# -----------------------------------------------------------------------------
# CLASS AND CATEGORY DEFINITIONS
# -----------------------------------------------------------------------------

CLASS_NAMES = [
    "Caries",               # 0
    "Crown",                # 1
    "Filling",              # 2
    "Implant",              # 3
    "Malaligned",           # 4
    "Mandibular Canal",     # 5
    "Missing teeth",        # 6
    "Periapical lesion",    # 7
    "Retained root",        # 8
    "Root Canal Treatment", # 9
    "Root Piece",           # 10
    "Impacted tooth",       # 11
    "Maxillary sinus",      # 12
    "Bone Loss",            # 13
    "Fractured teeth",      # 14
    "Permanent Teeth",      # 15
    "Supra Eruption",       # 16
    "TAD",                  # 17
    "Abutment",             # 18
    "Attrition",            # 19
    "Bone defect",          # 20
    "Gingival former",      # 21
    "Metal band",           # 22
    "Orthodontic brackets", # 23
    "Permanent retainer",   # 24
    "Post-core",            # 25
    "Plating",              # 26
    "Wire",                 # 27
    "Cyst",                 # 28
    "Root resorption",      # 29
    "Primary teeth",        # 30
]

CATEGORIES = {
    0: "Diseases",
    1: "Treatments",
    2: "Tooth Status",
    3: "Anatomy",
    4: "Orthodontics",
}

CLASS_TO_CATEGORY = {
    0:0, 7:0, 13:0, 28:0, 29:0, 20:0, 19:0,
    1:1, 2:1, 3:1,  9:1, 25:1, 26:1, 18:1, 21:1,
    6:2, 11:2, 8:2, 10:2, 14:2, 16:2, 4:2,
    5:3, 12:3, 15:3, 30:3,
    23:4, 27:4, 24:4, 17:4, 22:4,
}

CATEGORY_COLORS_RGB = {
    0: (220,  20,  60),   # Diseases     - Crimson
    1: ( 30, 144, 255),   # Treatments   - DodgerBlue
    2: ( 50, 205,  50),   # Tooth Status - LimeGreen
    3: (255, 165,   0),   # Anatomy      - Orange
    4: (148,   0, 211),   # Orthodontics - DarkViolet
}

NUM_CLASSES    = len(CLASS_NAMES)
NUM_CATEGORIES = len(CATEGORIES)
SPLITS         = ['train', 'valid', 'test']
IMG_EXTS       = {'.jpg', '.jpeg', '.png', '.bmp'}

# Verify path
print(f"Environment : {ENVIRONMENT}")
print(f"YOLO dir    : {YOLO_DIR}")
print(f"Exists      : {YOLO_DIR.exists()}")
if not YOLO_DIR.exists():
    print("WARNING: Path not found. Check ENVIRONMENT setting.")

In [ ]:
def get_image_paths(split):
    d = YOLO_DIR / split / "images"
    return sorted([p for p in d.iterdir() if p.suffix.lower() in IMG_EXTS])

def get_label_paths(split):
    d = YOLO_DIR / split / "labels"
    return sorted([p for p in d.iterdir() if p.suffix == ".txt"])

split_info = {}
for split in SPLITS:
    imgs = get_image_paths(split)
    lbls = get_label_paths(split)
    split_info[split] = {'images': imgs, 'labels': lbls}

total_imgs = sum(len(v['images']) for v in split_info.values())
total_lbls = sum(len(v['labels']) for v in split_info.values())

print("Dataset overview (original data, no modifications):")
print(f"  {'Split':<10} {'Images':>8} {'Labels':>8} {'Match':>8}")
print("  " + "-" * 38)
for split in SPLITS:
    imgs = len(split_info[split]['images'])
    lbls = len(split_info[split]['labels'])
    match = "yes" if imgs == lbls else "NO"
    print(f"  {split:<10} {imgs:>8} {lbls:>8} {match:>8}")
print("  " + "-" * 38)
print(f"  {'TOTAL':<10} {total_imgs:>8} {total_lbls:>8}")

In [ ]:
def parse_label_file(lbl_path):
    """Return list of class_ids found in a YOLO label file."""
    class_ids = []
    try:
        with open(lbl_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    try:
                        class_ids.append(int(parts[0]))
                    except ValueError:
                        pass
    except Exception:
        pass
    return class_ids


# Parse all label files across all splits
ann_rows = []
for split in SPLITS:
    for lbl_path in tqdm(split_info[split]['labels'], desc=f"Parsing {split}"):
        for cls_id in parse_label_file(lbl_path):
            if 0 <= cls_id < NUM_CLASSES:
                cat_id = CLASS_TO_CATEGORY.get(cls_id, -1)
                ann_rows.append({
                    'split'        : split,
                    'class_id'     : cls_id,
                    'class_name'   : CLASS_NAMES[cls_id],
                    'category_id'  : cat_id,
                    'category_name': CATEGORIES.get(cat_id, 'Unknown'),
                })

ann_df = pd.DataFrame(ann_rows)

print(f"Total annotations parsed : {len(ann_df):,}")
print(f"Unique classes found     : {ann_df['class_id'].nunique()} / {NUM_CLASSES}")
print(f"Avg annotations per image: {len(ann_df) / total_imgs:.1f}")

In [ ]:
# Instance count per class — printed table
class_counts = (
    ann_df.groupby(['class_id', 'class_name', 'category_name'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)

max_count = class_counts['count'].max()

print(f"  {'ID':>3}  {'Class':<25} {'Category':<15} {'Count':>7}  {'Bar'}")
print("  " + "-" * 70)
for _, row in class_counts.iterrows():
    bar = '#' * min(int(row['count'] / max_count * 30), 30)
    print(f"  {int(row['class_id']):>3}  {row['class_name']:<25} "
          f"{row['category_name']:<15} {int(row['count']):>7}  {bar}")

print()
print(f"  Most common  : {class_counts.iloc[0]['class_name']} ({int(class_counts.iloc[0]['count']):,})")
print(f"  Least common : {class_counts.iloc[-1]['class_name']} ({int(class_counts.iloc[-1]['count']):,})")
print(f"  Imbalance ratio: {max_count / max(class_counts['count'].min(), 1):.0f}x")

In [ ]:
# Class distribution chart
cat_name_to_id = {v: k for k, v in CATEGORIES.items()}

bar_colors = [
    tuple(c / 255 for c in CATEGORY_COLORS_RGB[
        CLASS_TO_CATEGORY.get(
            CLASS_NAMES.index(row['class_name']) if row['class_name'] in CLASS_NAMES else 0, 0
        )
    ])
    for _, row in class_counts.iterrows()
]

fig, ax = plt.subplots(figsize=(14, 10))
bars = ax.barh(
    class_counts['class_name'],
    class_counts['count'],
    color=bar_colors, edgecolor='white', linewidth=0.5
)
ax.set_title('Instance Count per Class (colored by category)', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Instances')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

for bar, count in zip(bars, class_counts['count']):
    ax.text(bar.get_width() + max_count * 0.005,
            bar.get_y() + bar.get_height() / 2,
            f'{int(count):,}', va='center', ha='left', fontsize=8)

legend_patches = [
    mpatches.Patch(
        color=tuple(c / 255 for c in CATEGORY_COLORS_RGB[cid]),
        label=cname
    )
    for cid, cname in CATEGORIES.items()
]
ax.legend(handles=legend_patches, loc='lower right', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Category-level summary
cat_summary = (
    ann_df.groupby(['category_id', 'category_name'])
    .agg(
        total_instances = ('class_id', 'count'),
        num_classes     = ('class_id', 'nunique'),
    )
    .reset_index()
    .sort_values('category_id')
)

print("Category summary:")
print(f"  {'ID':>3}  {'Category':<15} {'Classes':>8} {'Instances':>12} {'% of total':>12}")
print("  " + "-" * 56)
for _, row in cat_summary.iterrows():
    pct = row['total_instances'] / len(ann_df) * 100
    print(f"  {int(row['category_id']):>3}  {row['category_name']:<15} "
          f"{int(row['num_classes']):>8} {int(row['total_instances']):>12,} {pct:>11.1f}%")

print()
print("Classes inside each category:")
for cat_id, cat_name in CATEGORIES.items():
    classes_in_cat = [CLASS_NAMES[i] for i, c in CLASS_TO_CATEGORY.items() if c == cat_id]
    print(f"\n  [{cat_name}]")
    for cls_name in classes_in_cat:
        count = ann_df[ann_df['class_name'] == cls_name].shape[0]
        print(f"    {cls_name:<25} {count:>8,} instances")

In [ ]:
# Category distribution charts
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Category Distribution', fontsize=14, fontweight='bold')

# Pie chart
cat_counts = ann_df.groupby('category_name').size().reset_index(name='count')
cat_colors = [
    tuple(c / 255 for c in CATEGORY_COLORS_RGB[cat_name_to_id.get(n, 0)])
    for n in cat_counts['category_name']
]
wedges, texts, autotexts = axes[0].pie(
    cat_counts['count'],
    labels=cat_counts['category_name'],
    autopct='%1.1f%%',
    colors=cat_colors,
    startangle=90,
    pctdistance=0.80,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
)
for t in autotexts:
    t.set_fontsize(11)
    t.set_fontweight('bold')
axes[0].set_title('Share of all instances')
axes[0].legend(
    [f"{row['category_name']}: {row['count']:,}" for _, row in cat_counts.iterrows()],
    loc='lower center', bbox_to_anchor=(0.5, -0.12), ncol=2, fontsize=9
)

# Bar chart per category ordered by category_id
ordered_cats  = [CATEGORIES[i] for i in range(NUM_CATEGORIES)]
ordered_counts = [ann_df[ann_df['category_name'] == c].shape[0] for c in ordered_cats]
ordered_colors = [tuple(c / 255 for c in CATEGORY_COLORS_RGB[i]) for i in range(NUM_CATEGORIES)]

axes[1].bar(ordered_cats, ordered_counts, color=ordered_colors, edgecolor='white')
axes[1].set_title('Instance count per category')
axes[1].set_ylabel('Count')
axes[1].grid(axis='y', alpha=0.3)
for i, (cat, count) in enumerate(zip(ordered_cats, ordered_counts)):
    axes[1].text(i, count + max(ordered_counts) * 0.01,
                 f'{count:,}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
resolution_data = []
for split in SPLITS:
    for img_path in tqdm(split_info[split]['images'], desc=f"Reading {split} resolutions"):
        try:
            img = cv2.imread(str(img_path))
            if img is not None:
                h, w = img.shape[:2]
                resolution_data.append({
                    'split'       : split,
                    'width'       : w,
                    'height'      : h,
                    'aspect_ratio': round(w / h, 3),
                })
        except Exception:
            pass

res_df = pd.DataFrame(resolution_data)

print("Image resolution statistics:")
for split in SPLITS:
    s = res_df[res_df['split'] == split]
    if len(s) == 0:
        continue
    print(f"\n  [{split}]")
    print(f"    Width  : min={s['width'].min()}  max={s['width'].max()}  mean={s['width'].mean():.0f}")
    print(f"    Height : min={s['height'].min()}  max={s['height'].max()}  mean={s['height'].mean():.0f}")
    print(f"    Aspect : min={s['aspect_ratio'].min():.2f}  max={s['aspect_ratio'].max():.2f}  mean={s['aspect_ratio'].mean():.2f}")

In [ ]:
colors_map = {'train': 'steelblue', 'valid': 'darkorange', 'test': 'seagreen'}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Image Resolution Analysis', fontsize=14, fontweight='bold')

for split in SPLITS:
    s = res_df[res_df['split'] == split]
    axes[0].hist(s['width'],  bins=20, alpha=0.6, label=split, color=colors_map[split])
    axes[1].hist(s['height'], bins=20, alpha=0.6, label=split, color=colors_map[split])
    axes[2].scatter(s['width'], s['height'], alpha=0.4, s=10,
                    label=split, c=colors_map[split])

axes[0].set_title('Width Distribution');  axes[0].set_xlabel('Width (px)');  axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set_title('Height Distribution'); axes[1].set_xlabel('Height (px)'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[2].set_title('Width vs Height');     axes[2].set_xlabel('Width');       axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Which classes are present in each split
print("Class coverage per split:")
for split in SPLITS:
    present = set(ann_df[ann_df['split'] == split]['class_id'].unique())
    missing = [CLASS_NAMES[i] for i in range(NUM_CLASSES) if i not in present]
    print(f"\n  [{split}] {len(present)}/{NUM_CLASSES} classes present")
    if missing:
        print(f"    Missing: {missing}")
    else:
        print(f"    All classes present.")

print()

# Per-class count broken down by split
pivot = (
    ann_df.groupby(['class_name', 'split'])
    .size()
    .unstack(fill_value=0)
    .reindex(CLASS_NAMES)
    .fillna(0)
    .astype(int)
)
pivot['TOTAL'] = pivot.sum(axis=1)
pivot = pivot.sort_values('TOTAL', ascending=False)

print("Instance count per class per split:")
print(f"  {'Class':<25}", end="")
for col in pivot.columns:
    print(f"  {col:>8}", end="")
print()
print("  " + "-" * (25 + len(pivot.columns) * 10))
for cls_name, row in pivot.iterrows():
    print(f"  {cls_name:<25}", end="")
    for col in pivot.columns:
        print(f"  {int(row[col]):>8}", end="")
    print()

In [ ]:
# Final summary
print("=" * 55)
print("DATA UNDERSTANDING SUMMARY")
print("=" * 55)
print(f"  Total images      : {total_imgs:,}")
print(f"    train           : {len(split_info['train']['images']):,}")
print(f"    valid           : {len(split_info['valid']['images']):,}")
print(f"    test            : {len(split_info['test']['images']):,}")
print(f"  Total annotations : {len(ann_df):,}")
print(f"  Classes           : {NUM_CLASSES}")
print(f"  Categories        : {NUM_CATEGORIES}")
print(f"  Avg ann/image     : {len(ann_df)/total_imgs:.1f}")
print(f"  Imbalance ratio   : {max_count / max(class_counts['count'].min(),1):.0f}x")
print(f"  Most common class : {class_counts.iloc[0]['class_name']} ({int(class_counts.iloc[0]['count']):,})")
print(f"  Least common class: {class_counts.iloc[-1]['class_name']} ({int(class_counts.iloc[-1]['count']):,})")
print("=" * 55)